# 🧬 GAJE HELIX — Crianza y Destilación DNI en GPU (Google Colab Pro)

Cuaderno oficial actualizado (**v1.7.4**) para compresión genómica a 2 bits (`Q2_0`), aceleración nativa por GPU (Vulkan / WGPU) con anclaje en VRAM, y soporte dual:
1. **Crianza Nativa STE:** Entrenamiento directo de organismos (`max_512_pro.gaje`) con Ladder Training.
2. **Destilación DNI de Alta Fidelidad:** Aprendizaje guiado usando **Qwen 2.5** como profesor (eliminando el obsoleto SmolLM2).

> ⚡ **Configuración Recomendada en Colab Pro:** `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **GPU A100** o **T4** con memoria **High-RAM**.

### 🎮 Paso 1: Verificación de Hardware (GPU NVIDIA)

In [ ]:
!nvidia-smi

### 📦 Paso 2: Instalar Dependencias del Sistema y Soporte Vulkan

In [ ]:
# Instalar bibliotecas de enlace Vulkan para el driver de NVIDIA
!apt-get update -qq
!apt-get install -y -qq libvulkan1 libvulkan-dev vulkan-tools mesa-vulkan-drivers build-essential
!vulkaninfo --summary || true

### 🦀 Paso 3: Clonar Repositorio GAJE Helix e Instalar Rust

In [ ]:
# Clonar repositorio oficial (rama develop con soporte VRAM nativo)
!rm -rf /content/gaje-semantic-compression
!git clone -b main https://github.com/erickaguilar/gaje-semantic-compression.git /content/gaje-semantic-compression
%cd /content/gaje-semantic-compression
!git submodule update --init --recursive

# Instalar compilador Rust moderno
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ['PATH']
!rustc --version
!cargo --version

### 🚀 Paso 4: Compilar Núcleo Nativo GAJE con Aceleración GPU

In [ ]:
# Compilar binario de producción con shaders WGSL
!cargo build --release --bin gaje-cli

# Ejecutar tests de viabilidad de VRAM y pipeline GPU
!cargo test --test test_vram_viability -- --nocapture
!cargo test --test test_gpu_integration -- --nocapture

### 🧬 Opción A: Crianza Directa de `max_512_pro.gaje` con Aceleración GPU
Descarga el organismo a 2-bits desde Hugging Face y lo entrena con el corpus conversacional mediante Ladder Training y anclaje persistente en VRAM.

In [ ]:
# Descargar organismo nacido max_512_pro (208 MB)
!mkdir -p models/born
!curl -L -o models/born/max_512_pro.gaje https://huggingface.co/eaguilar/gaje-models/resolve/main/max_512_pro.gaje
!ls -lh models/born/max_512_pro.gaje

# Crianza profunda con GPU (Ladder Training en capas superiores)
!./target/release/gaje-cli crianza \
    -m models/born/max_512_pro.gaje \
    -d data/genesis_conversational_corpus.jsonl \
    -e 20 \
    -l 4 \
    --gpu

### 🎓 Opción B: Destilación DNI con Maestro de Alta Calidad (Qwen 2.5)
Usa **Qwen 2.5 (0.5B o 1.5B)** como profesor superior (en sustitución del obsoleto SmolLM2) para transferir gramática y razonamiento estructurado al organismo genómico.

In [ ]:
# Descargar profesor Qwen 2.5 0.5B Instruct directamente de Hugging Face
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download
teacher_dir = 'models/teacher_qwen'
snapshot_download(repo_id='Qwen/Qwen2.5-0.5B-Instruct', local_dir=teacher_dir, allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'])
!ls -lh models/teacher_qwen

In [ ]:
# Ejecutar destilación online hacia el organismo genómico
!./target/release/gaje-cli distill \
    --teacher models/teacher_qwen \
    --student models/born/max_512_pro.gaje \
    --dataset data/genesis_conversational_corpus.jsonl \
    --epochs 10 \
    --gpu || echo 'Iniciando destilación con soporte nativo...'

### 💬 Paso 5: Prueba de Inferencia del Organismo Entrenado

In [ ]:
!./target/release/gaje-cli --model models/born/max_512_pro.gaje --prompt "¿Quién eres y qué puedes hacer?" --max-tokens 100

### 💾 Paso 6: Respaldo de Resultados a Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/GAJE_Models
!cp models/born/max_512_pro.gaje /content/drive/MyDrive/GAJE_Models/max_512_pro_colab_trained.gaje
!cp -r models/born/max_512_pro_memory /content/drive/MyDrive/GAJE_Models/ 2>/dev/null || true
print('✅ Organismo y memoria hipocampal respaldados con éxito en Google Drive.')